In [ ]:
# ! pip install google-genai

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import threading
from PIL import Image

import dotenv
dotenv.load_dotenv()

place_list = {
      "restaurant": 1000,
      "cafe": 300,
      "fast_food": 594,
      "convenience_store": 500,
      "parking_entrance": 450,
      "bar": 345,
      "bank": 217,
      "atm": 185,
      "bicycle_rental": 180,
      "pharmacy": 125,
      "toilets": 106,
      "theatre": 66,
      "library": 56,
      "dentist": 51,
      "school": 20,
      "post_office": 40,
      "bureau_de_change": 38,
      "doctors": 31,
      "bicycle_repair_station": 29,
      "clinic": 27,
      "community_centre": 26,
      "car_rental": 9,
      "police": 20,
      "arts_centre": 20,
      "cinema": 17,
      "kindergarten": 14,
      "university": 12,
      "coworking_space": 11,
      "lab": 11,
}

place_type = "restaurant"

prompt_rgb = \
"""A highly detailed, photorealistic orthographic front elevation view of a {place_type} storefront facade, intended for use as a flat texture in a simulation.

Layout:
- ABSOLUTELY NO sidewalk, pavement, street furniture, or surrounding building structure is visible.
- The facade occupies the entire frame, WITHOUT ANY borders or outer walls or background.
- The outer left and right edges of the storefront's main structural frame and glazing system align exactly with the left and right borders of the image canvas. No brick wall, siding, or adjacent building material is visible on the sides.
- The door could be on the left, right, or in the middle.

Signage & Branding: If the store type is general like restaurant, decide a random sub-category (e.g., different food types) for the store first. The main overhead signage prominently features a creative, non-infringing fake brand name. If the store type is a public facility, just use the business type as the brand name instead of a fake brand name. Integrated into the main sign and window decals are easily recognizable, large icons **relevant to the business type** (e.g., a stylized cupcake and whisk for a cafe; a retro hanger and boot for a vintage clothing store; a bold red cross and heart for a clinic). The font style should match the business theme.

Materials & Details: The facade is constructed from photorealistic, high-quality textures. The primary materials should match the business theme. Add realistic details like mortar lines, wood grain, or slight weathering appropriate for the materials.

Choose one of the following style and materials according to the business theme: classic: weathered red brick with dark wood trim; industrial: polished concrete with brushed metal panels; rustic: cream stucco with distressed timber beams; modern: glossy white panels with seamless glass; art_deco: black marble with geometric gold brass patterns; japanese: light cypress wood slats with stone base; victorian: ornate painted wood with decorative molding; cyberpunk: dark metal plating with neon signage strips; mediterranean: rough white plaster with terracotta tiles; scandinavian: vertical pale wood cladding with large windows; mid_century: warm teak wood with stone accents; retro: chrome plating with red enamel panels; gothic: dark grey stone with pointed arches; green: living plant wall with cedar framing; budget: beige ceramic tiles with plain aluminum window frames; utility: split-face concrete block with heavy steel doors

Lighting & Glass Constraints: The entire scene is lit with completely flat, neutral, even light (similar to an ambient occlusion pass) to ensure there are no harsh shadows or directional highlights. All windows and glass doors must be fully transparent, revealing a well lit lively interior space behind them. The windows must be completely NON-REFLECTIVE: there should be no reflections of an outside environment on the glass.
"""

prompt_depth = \
"""A purely mathematical, flat-shaded z-depth map of a {place_type} storefront. Darker gray represent higher depth values.

Style: Technical diagram, flat vector art style. UNLIT. No lighting, no shading, no ambient occlusion.

Detail: The depth map should be as detailed as geometry of objects in the RGB image.

Coloring Rules (The "Block" Look):
- Quantized Depth: Use distinct, solid flat blocks of gray color for different depth planes.
- Background: Pure white (RGB 255, 255, 255).
- Windows/Doors: Rendered as SOLID opaque planes. They must be a single flat color code, flush with the frame. Do not render depth for interior objects.
- Order of depth: sign (white) <= wall near sign <= wall near window <= window <= door (black).
- The sign must be plain gray color, absolutely NO text or icon.

Visual Definition: Sharp, pixel-perfect hard edges. No gradients, no soft shadows, no dirt, no noise. The image should look like a posterized segmentation map, not a 3D render.
"""

place_type_list = []
for place_type, place_number in place_list.items():
    store_front_number = max(place_number//100, 3)
    print(f"Generating {store_front_number} {place_type}")
    for i in range(store_front_number):
        place_type_list.append((place_type, i))
print(f"Generating {len(place_type_list)} storefronts")


def generate_image(prompt_content):
    from google import genai
    from google.genai import types
    from PIL import Image

    client = genai.Client()

    response = client.models.generate_content(
        model="gemini-3-pro-image-preview",
        contents=prompt_content,
        config=types.GenerateContentConfig(
            tools=[{"google_search": {}}],
            image_config=types.ImageConfig(
                aspect_ratio="16:9",
                image_size="2K"
            ),
        )
    )

    for part in response.parts:
        if part.text is not None:
            print(part.text)
        elif part.inline_data is not None:
            image = part.as_image()
    return image
    

def generate_and_save_storefront(place_type, i, notes={}):
    place_id = f"{place_type}_{i}"
    folder_path = f"data/{place_id}"
    rgb_path = f"{folder_path}/rgb.png"
    depth_path = f"{folder_path}/depth.png"
    os.makedirs(folder_path, exist_ok=True)
    if not os.path.exists(rgb_path):
        print(f"Generating rgb for {place_type} {i}")
        prompt = prompt_rgb.format(place_type=place_type)
        if place_id in notes:
            prompt = prompt + "\nRequirements: " + notes[place_id]
            print(f"Notes for {place_id}: {notes[place_id]}")
        rgb_image = generate_image(prompt)
        rgb_image.save(f"{folder_path}/rgb.png")
    if not os.path.exists(depth_path):
        print(f"Generating depth for {place_type} {i}")
        prompt = prompt_depth.format(place_type=place_type)
        rgb_image = Image.open(rgb_path)
        depth_image = generate_image([prompt, rgb_image])
        depth_image.save(f"{folder_path}/depth.png")

In [ ]:
# prompt = prompt_rgb.format(place_type="fast_food")
# image = generate_image(prompt)
# image.save("generated_image.png")

# generate_and_save_storefront("fast_food", 5)

In [ ]:

notes = {
    "restaurant_7": "Generate a chinese sichuan restaurant",
    "restaurant_9": "Generate a chinese hot pot restaurant",
    "school_0": "Generate a elementary school",
    "school_1": "Generate a middle school with a fake name",
    "school_2": "Generate a high school with a fake name",
    "fast_food_2": "Generate a malatang shop (no dragon or typical chinese elements)",
    "fast_food_3": "Generate a icecream and drinks shop",
    "fast_food_4": "Generate a chinese dumplings restaurant (no dragon or typical chinese elements)",
    "lab_0": "Generate a robotics lab called 'CURLY Robotics Lab'",
    "lab_2": "Generate a computer science lab with a lot of servers",
    "university_0": "Generate a university entrance with a big clock tower. Named University of xxx (come up with a fake name)",
    "university_1": "Generate a university entrance with security guards. Named University of xxx (come up with a fake name)",
    "university_2": "Generate a university entrance that can see classrooms. Named University of xxx (come up with a fake name)",
}

import queue
threads = []
place_type_queue = queue.Queue()
for place_type, i in place_type_list:
    place_type_queue.put((place_type, i))
while not place_type_queue.empty():
    place_type, i = place_type_queue.get()
    t = threading.Thread(target=generate_and_save_storefront, args=(place_type, i, notes))
    t.start()
    threads.append(t)

for t in threads:
    t.join()


In [ ]:
"lab_2" in notes

In [ ]:
!python batch_usd_export.py